In [156]:
import pandas as pd
import os

base_directory = os.getcwd()
relative_folder_path = os.path.join(base_directory, "allDatasets", "customerChurn")
churn_dataset = os.path.join(relative_folder_path, "customerChurn.csv")
df = pd.read_csv(churn_dataset)
df.drop('customerID', axis = 'columns', inplace=True)
print(df.head())
df.dtypes



   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No  

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [157]:
# Converting the dtype of totalCharges to numerical
# pd.to_numeric(df.TotalCharges)
# I faced an error: ValueError: Unable to parse string " " at position 488
# For this I will find out the null values in the totalCharges column
pd.to_numeric(df.TotalCharges, errors='coerce').isnull()
# Now in order to find out what those values are, I will pass this column to the dataframe as index.
df[pd.to_numeric(df.TotalCharges, errors='coerce').isnull()] # Total 11 rows with null values
updatedDf = df[df.TotalCharges!= " "] # Removing the null value rows and updating the DF
updatedDf.TotalCharges = pd.to_numeric(updatedDf.TotalCharges)
updatedDf.dtypes


C:\Users\arrey\AppData\Local\Temp\ipykernel_8316\4011514209.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  updatedDf.TotalCharges = pd.to_numeric(updatedDf.TotalCharges)


gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

# Encoding the values:

In [158]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Identify categorical and numerical columns
categoricalCols = updatedDf.select_dtypes(include=['object']).columns
numericalCols = updatedDf.select_dtypes(include=['int64', 'float64']).columns

# for cols in updatedDf:
#     if updatedDf[cols].dtypes == "object":
#         print(f'{cols}: {updatedDf[cols].unique()}')

updatedDf.replace('No phone service', 'No', inplace=True)
updatedDf.replace('No internet service', 'No', inplace=True)

for cols in categoricalCols:
    print(f'{cols}: {updatedDf[cols].unique()}')

affirmativeValues = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                    'PaperlessBilling', 'Churn']

for cols in affirmativeValues:
    updatedDf[cols].replace({'Male': 1, "Female": 0, 'Yes': 1, "No": 0}, inplace = True)


oneHotEncodingList = ['InternetService', 'Contract', 'PaymentMethod']
updatedDf1 = pd.get_dummies(data = updatedDf, columns = oneHotEncodingList)
# updatedDf1
multiValList = [
        'InternetService_DSL',
        'InternetService_No',
        'InternetService_Fiber optic', 
        'Contract_Month-to-month', 
        'Contract_One year', 
        'Contract_Two year', 
        'PaymentMethod_Bank transfer (automatic)', 
        'PaymentMethod_Credit card (automatic)', 
        'PaymentMethod_Electronic check', 
        'PaymentMethod_Mailed check'
    ]
for col in multiValList:
    updatedDf1[col] = updatedDf1[col].replace({True: 1, False: 0})
updatedDf1.shape





gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes']
OnlineBackup: ['Yes' 'No']
DeviceProtection: ['No' 'Yes']
TechSupport: ['No' 'Yes']
StreamingTV: ['No' 'Yes']
StreamingMovies: ['No' 'Yes']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn: ['No' 'Yes']


C:\Users\arrey\AppData\Local\Temp\ipykernel_8316\1372048345.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  updatedDf.replace('No phone service', 'No', inplace=True)
C:\Users\arrey\AppData\Local\Temp\ipykernel_8316\1372048345.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  updatedDf.replace('No internet service', 'No', inplace=True)
C:\Users\arrey\AppData\Local\Temp\ipykernel_8316\1372048345.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because 

(7032, 27)

In [159]:
scalingCols = ['tenure', 'MonthlyCharges', 'TotalCharges']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
updatedDf1[scalingCols] = scaler.fit_transform(updatedDf1[scalingCols])
updatedDf1

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,0.000000,0,0,0,1,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,0.464789,1,0,1,0,1,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,0.014085,1,0,1,1,0,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,0.619718,0,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,0.014085,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,0.323944,1,1,1,0,1,...,1,0,0,0,1,0,0,0,0,1
7039,0,0,1,1,1.000000,1,1,0,1,1,...,0,1,0,0,1,0,0,1,0,0
7040,0,0,1,1,0.140845,0,0,1,0,0,...,1,0,0,1,0,0,0,0,1,0
7041,1,1,1,0,0.042254,1,1,0,0,0,...,0,1,0,1,0,0,0,0,0,1


In [160]:
filePathExcel = os.path.join(base_directory, "datasetReadyForML.xlsx")
updatedDf1.to_excel(filePathExcel, index= True)

mlReady = updatedDf1
mlReady

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,0.000000,0,0,0,1,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,0.464789,1,0,1,0,1,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,0.014085,1,0,1,1,0,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,0.619718,0,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,0.014085,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,0.323944,1,1,1,0,1,...,1,0,0,0,1,0,0,0,0,1
7039,0,0,1,1,1.000000,1,1,0,1,1,...,0,1,0,0,1,0,0,1,0,0
7040,0,0,1,1,0.140845,0,0,1,0,0,...,1,0,0,1,0,0,0,0,1,0
7041,1,1,1,0,0.042254,1,1,0,0,0,...,0,1,0,1,0,0,0,0,0,1


In [161]:
x = mlReady.drop('Churn', axis = 'columns')
y = mlReady['Churn']

from sklearn.model_selection import train_test_split
xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size = 0.3, random_state = 12)

In [164]:
xTrain.shape

(4922, 26)

In [166]:
xTest.shape

(2110, 26)

In [168]:
import tensorflow as tf
from tensorflow import keras

AttributeError: `np.complex_` was removed in the NumPy 2.0 release. Use `np.complex128` instead.